# MiseCast Exploration Notebook
#### Team SVNAD

A notebook file to examine components such as data and sanity checks.

---

In [4]:
import pandas as pd
import numpy as np
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))

from sklearn.ensemble import HistGradientBoostingRegressor

from utils import data_handling as dh

In [2]:
DATA_DIR = Path.cwd().parent / "data"

menu_items = dh.load_menu_items(DATA_DIR / "menu_items.csv")
menu_variants = dh.load_menu_variants(DATA_DIR / "menu_variants.csv")
menu_recipes = dh.load_menu_recipes(DATA_DIR / "menu_recipes.csv")
ingredient_master = dh.load_ingredient_master(DATA_DIR / "ingredient_master.csv")
sales = dh.load_historical_sales(DATA_DIR / "historical_sales.csv")

dh.check_joins(menu_variants, menu_recipes, ingredient_master, sales)

All joins check out: recipes -> variants -> sales, and recipes -> ingredient_master.


In [10]:
def load_calendar_context(path):
    df = pd.read_csv(path, sep=";")
    df.columns = [c.strip() for c in df.columns]
    df["date"] = pd.to_datetime(df["date"])
    return df


def load_reservations(path):
    df = pd.read_csv(path, sep=";")
    df.columns = [c.strip() for c in df.columns]
    df["date"] = pd.to_datetime(df["date"])
    return df


def load_weather(path):
    df = pd.read_csv(path, sep=";")
    df.columns = [c.strip() for c in df.columns]
    df["date"] = pd.to_datetime(df["date"])
    return df


def load_local_events(path):
    df = pd.read_csv(path, sep=";")
    df.columns = [c.strip() for c in df.columns]
    df["event_date"] = pd.to_datetime(df["event_date"])
    return df


def load_promotion_history(path):
    df = pd.read_csv(path, sep=";")
    df.columns = [c.strip() for c in df.columns]
    df["promotion_date"] = pd.to_datetime(df["promotion_date"])
    return df


def load_current_inventory(path):
    df = pd.read_csv(path, sep=";")
    df.columns = [c.strip() for c in df.columns]
    df["inventory_as_of_date"] = pd.to_datetime(df["inventory_as_of_date"])
    return df


def load_inventory_batches_expiry(path):
    df = pd.read_csv(path, sep=";")
    df.columns = [c.strip() for c in df.columns]
    df["received_date"] = pd.to_datetime(df["received_date"])
    df["expiry_date"] = pd.to_datetime(df["expiry_date"])
    return df


calendar = load_calendar_context(DATA_DIR / "calendar_context.csv")
reservations = load_reservations(DATA_DIR / "reservations.csv")
weather = load_weather(DATA_DIR / "weather.csv")
events = load_local_events(DATA_DIR / "local_events.csv")
promotions = load_promotion_history(DATA_DIR / "promotion_history.csv")
current_inventory = load_current_inventory(DATA_DIR / "current_inventory.csv")
batches = load_inventory_batches_expiry(DATA_DIR / "inventory_batches_expiry.csv")

### 1. Feature Engineering

In [3]:
def build_feature_table(sales, menu_variants, calendar, reservations, weather, events, promotions):
    # aggregate raw sales rows (per channel) up to one row per date/service/variant
    agg = (sales.groupby(["date", "service_period", "menu_variant_id", "menu_item_id"], as_index=False)
                 .agg(quantity_sold=("quantity_sold", "sum")))

    # drop variants the data itself flags as not trainable (e.g. the inactive MI-067)
    eligible = set(menu_variants.loc[menu_variants.demand_model_training_eligible, "menu_variant_id"])
    df = agg[agg.menu_variant_id.isin(eligible)].reset_index(drop=True)

    cal_cols = ["date", "day_of_week", "month", "season", "is_weekend", "is_public_holiday",
                "is_school_holiday", "special_occasion_demand_factor"]
    df = df.merge(calendar[cal_cols], on="date", how="left")

    # only "actual" rows for training -- "forecast" rows are for future prediction, not history
    res_actual = reservations[reservations.record_type == "actual"][["date", "service_period", "reserved_covers"]]
    df = df.merge(res_actual, on=["date", "service_period"], how="left")
    df["reserved_covers"] = df["reserved_covers"].fillna(df["reserved_covers"].median())

    wx_actual = weather[weather.record_type == "actual"][
        ["date", "average_temperature_c", "rain_probability_pct", "is_hot_day", "is_cold_day"]]
    df = df.merge(wx_actual, on="date", how="left")

    df = df.merge(menu_variants[["menu_variant_id", "category", "portion_size"]], on="menu_variant_id", how="left")

    def event_score(row):
        relevant = events[events.service_period_impacted.isin([row.service_period, "Both"])]
        if relevant.empty:
            return 0.0
        day_diff = (relevant.event_date - row.date).abs().dt.days
        weight = relevant.demand_relevance_score / (1 + relevant.distance_from_restaurant_km)
        if (day_diff <= 0).any():
            return float(weight[day_diff <= 0].max())
        nearest = day_diff.idxmin()
        return float(weight.loc[nearest] * np.exp(-day_diff.loc[nearest] / 2.0))

    df["event_proximity"] = df.apply(event_score, axis=1)

    promo_keys = set(zip(promotions.promotion_date, promotions.service_period, promotions.menu_variant_id))
    df["promotion_active"] = df.apply(
        lambda r: (r.date, r.service_period, r.menu_variant_id) in promo_keys, axis=1)

    # matching-weekday lag/rolling (per variant+service_period+weekday) -- this IS the baseline too
    df = df.sort_values(["menu_variant_id", "service_period", "day_of_week", "date"]).reset_index(drop=True)
    g_weekday = df.groupby(["menu_variant_id", "service_period", "day_of_week"])["quantity_sold"]
    df["lag_7d"] = g_weekday.shift(1)
    df["rolling_matching_weekday_avg"] = g_weekday.transform(lambda s: s.shift(1).rolling(4, min_periods=1).mean())

    # short-term momentum across consecutive dates, same variant+service_period
    df = df.sort_values(["menu_variant_id", "service_period", "date"]).reset_index(drop=True)
    g_series = df.groupby(["menu_variant_id", "service_period"])["quantity_sold"]
    df["recent_momentum"] = g_series.transform(lambda s: s.diff().rolling(3, min_periods=1).mean().shift(1))

    overall_mean = df.groupby("menu_variant_id")["quantity_sold"].transform("mean")
    df["lag_7d"] = df["lag_7d"].fillna(overall_mean)
    df["rolling_matching_weekday_avg"] = df["rolling_matching_weekday_avg"].fillna(overall_mean)
    df["recent_momentum"] = df["recent_momentum"].fillna(0.0)

    return df

### 2. The Pooled Model

In [8]:
CATEGORICAL = ["menu_variant_id", "menu_item_id", "category", "portion_size",
               "service_period", "day_of_week", "month", "season"]
NUMERIC = ["is_weekend", "is_public_holiday", "is_school_holiday", "special_occasion_demand_factor",
           "reserved_covers", "average_temperature_c", "rain_probability_pct", "is_hot_day",
           "is_cold_day", "event_proximity", "promotion_active", "lag_7d",
           "rolling_matching_weekday_avg", "recent_momentum"]
FEATURES = CATEGORICAL + NUMERIC


def train_quantile_models(train_df, target="quantity_sold"):
    df = train_df.copy()
    for c in CATEGORICAL:
        df[c] = df[c].astype("category")
    for c in NUMERIC:
        if df[c].dtype == bool:
            df[c] = df[c].astype(int)

    models = {}
    for name, q in {"p50": 0.5, "p90": 0.9}.items():
        m = HistGradientBoostingRegressor(
            loss="quantile", quantile=q, max_depth=6, max_iter=300,
            min_samples_leaf=15, learning_rate=0.06, random_state=42,
            categorical_features="from_dtype",
        )
        m.fit(df[FEATURES], df[target])
        models[name] = m
    return models


def predict(models, feature_df):
    out = feature_df.copy()
    for c in CATEGORICAL:
        out[c] = out[c].astype("category")
    for c in NUMERIC:
        if out[c].dtype == bool:
            out[c] = out[c].astype(int)
    out["p50"] = models["p50"].predict(out[FEATURES])
    out["p90"] = np.maximum(models["p90"].predict(out[FEATURES]), out["p50"])
    return out

### 3. Evaluation vs Required Baseline

In [6]:
def wape(actual, forecast):
    actual, forecast = np.asarray(actual, float), np.asarray(forecast, float)
    return np.abs(actual - forecast).sum() / actual.sum()


def evaluate(holdout_with_preds, target="quantity_sold"):
    holdout_with_preds["baseline"] = holdout_with_preds["rolling_matching_weekday_avg"]
    overall = {
        "model_wape": wape(holdout_with_preds[target], holdout_with_preds["p50"]),
        "baseline_wape": wape(holdout_with_preds[target], holdout_with_preds["baseline"]),
    }
    by_category = (holdout_with_preds.groupby("category", observed=True)
                   .apply(lambda g: pd.Series({
                       "model_wape": wape(g[target], g["p50"]),
                       "baseline_wape": wape(g[target], g["baseline"]),
                   })))
    by_category["beats_baseline"] = by_category.model_wape < by_category.baseline_wape
    return overall, by_category

### 4. Future-Window Forecast

In [7]:
def build_future_feature_table(hist, menu_variants, calendar, reservations, weather, events, promotions):
    last_actual_date = hist["date"].max()
    future_dates = calendar.loc[calendar["date"] > last_actual_date, "date"]
    if future_dates.empty:
        raise ValueError("No future dates found beyond the last actual sales date.")

    eligible = menu_variants.loc[menu_variants.demand_model_training_eligible,
                                  ["menu_variant_id", "menu_item_id", "category", "portion_size"]]
    service_periods = sorted(hist["service_period"].unique())

    skeleton = (
        eligible.assign(key=1)
        .merge(pd.DataFrame({"service_period": service_periods, "key": 1}), on="key")
        .merge(pd.DataFrame({"date": future_dates, "key": 1}), on="key")
        .drop(columns="key")
    )

    cal_cols = ["date", "day_of_week", "month", "season", "is_weekend", "is_public_holiday",
                "is_school_holiday", "special_occasion_demand_factor"]
    df = skeleton.merge(calendar[cal_cols], on="date", how="left")

    res_fcst = reservations[reservations.record_type == "forecast"][["date", "service_period", "reserved_covers"]]
    df = df.merge(res_fcst, on=["date", "service_period"], how="left")
    df["reserved_covers"] = df["reserved_covers"].fillna(hist["reserved_covers"].median())

    wx_fcst = weather[weather.record_type == "forecast"][
        ["date", "average_temperature_c", "rain_probability_pct", "is_hot_day", "is_cold_day"]]
    df = df.merge(wx_fcst, on="date", how="left")
    for c in ["average_temperature_c", "rain_probability_pct"]:
        df[c] = df[c].fillna(hist[c].median())
    for c in ["is_hot_day", "is_cold_day"]:
        df[c] = df[c].fillna(False)

    def event_score(row):
        relevant = events[events.service_period_impacted.isin([row.service_period, "Both"])]
        if relevant.empty:
            return 0.0
        day_diff = (relevant.event_date - row.date).abs().dt.days
        weight = relevant.demand_relevance_score / (1 + relevant.distance_from_restaurant_km)
        if (day_diff <= 0).any():
            return float(weight[day_diff <= 0].max())
        nearest = day_diff.idxmin()
        return float(weight.loc[nearest] * np.exp(-day_diff.loc[nearest] / 2.0))

    df["event_proximity"] = df.apply(event_score, axis=1)

    promo_keys = set(zip(promotions.promotion_date, promotions.service_period, promotions.menu_variant_id))
    df["promotion_active"] = df.apply(
        lambda r: (r.date, r.service_period, r.menu_variant_id) in promo_keys, axis=1)

    tail_stats = (
        hist.sort_values("date")
            .groupby(["menu_variant_id", "service_period", "day_of_week"])
            .agg(lag_7d=("quantity_sold", "last"),
                 rolling_matching_weekday_avg=("quantity_sold", lambda s: s.tail(4).mean()))
            .reset_index()
    )
    momentum_last = (
        hist.sort_values("date")
            .groupby(["menu_variant_id", "service_period"])["recent_momentum"]
            .last()
            .reset_index()
    )
    df = df.merge(tail_stats, on=["menu_variant_id", "service_period", "day_of_week"], how="left")
    df = df.merge(momentum_last, on=["menu_variant_id", "service_period"], how="left")

    overall_mean = hist.groupby("menu_variant_id")["quantity_sold"].mean()
    fallback = df["menu_variant_id"].map(overall_mean)
    df["lag_7d"] = df["lag_7d"].fillna(fallback)
    df["rolling_matching_weekday_avg"] = df["rolling_matching_weekday_avg"].fillna(fallback)
    df["recent_momentum"] = df["recent_momentum"].fillna(0.0)

    return df

In [9]:
models = train_quantile_models(hist)                                    # hist = build_feature_table(...) output
future = build_future_feature_table(hist, menu_variants, calendar, reservations, weather, events, promotions)
forecast = predict(models, future)

NameError: name 'hist' is not defined